# NB1 — Baseline: variance propagation and the α = 1/√2 control

**Purpose.**
1. Reproduce the textbook claims this project builds on: with He initialisation a residual
   block doubles the activation variance (UDL §11.3), and rescaling by 1/√2 keeps it
   constant (UDL Fig. 11.6b).
2. Derive and check the initialisation σ²(α) = 2(1−α²)/D, and the per-block calibration that
   holds the forward variance fixed for any skip weight α, including a check of the backward pass.
3. Train the control condition used in NB2 (32 blocks, α = 1/√2) with ≥ 3 seeds and analyse
   its variability.

**Contents.** 0 Setup · 1 Model · 2 Reproducing variance doubling · 3 Derivation of σ²(α) ·
4 Finite-width spread and calibration · 5 Training the baseline · 6 Variability ·
7 Decisions carried into NB2

In [ ]:
QUICK = True  # Submission target: cached runs in under five minutes on the lab kernel.

## 0. Setup

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import config as C
from utils import (env_report, get_device, set_seed, make_generator, load_fashion_mnist,
                   iterate_minibatches, evaluate, save_result, result_exists, load_results,
                   summarise, plot_mean_sd, plot_seed_points)

T0 = time.time()
ENV = env_report()
DEVICE = get_device()
DATA = load_fashion_mnist(device=DEVICE, n_val=C.N_VAL, split_seed=C.SPLIT_SEED,
                          n_calib=C.N_CALIB, calib_seed=C.CALIB_SEED)
print({k: tuple(v.shape) for k, v in DATA.items() if torch.is_tensor(v)})

## 1. Model

Pre-activation residual block (UDL Fig. 11.5b): `h ← α·h + W·ReLU(h)`, no bias and no
normalisation, as specified in the design.

_TODO (owner): explain in a few sentences why the activation comes before the linear layer,
and why bias and normalisation are left out._

In [ ]:
class ResMLP(nn.Module):
    """Pre-activation residual MLP.

    input  : Linear(784 -> D, bias=False)                  -> h_0
    block l: h_{l+1} = alpha * h_l + W_l(ReLU(h_l))         W_l = Linear(D -> D, bias=False)
    output : Linear(D -> 10, bias=False) applied to h_L      (logits)

    forward(x, return_hidden=False) -> logits, or (logits, [h_0, ..., h_L]) if return_hidden.
    """

    def __init__(self, depth: int, width: int = C.WIDTH, alpha: float = 0.0):
        super().__init__()
        self.alpha = alpha
        self.input_layer = nn.Linear(C.IN_DIM, width, bias=False)
        self.blocks = nn.ModuleList(
            nn.Linear(width, width, bias=False) for _ in range(depth)
        )
        self.output_layer = nn.Linear(width, C.N_CLASSES, bias=False)

    def forward(self, x: torch.Tensor, return_hidden: bool = False):
        x = x.flatten(start_dim=1)
        h = self.input_layer(x)

        if return_hidden:
            hidden = [h]

        for block in self.blocks:
            h = self.alpha * h + block(torch.relu(h))
            if return_hidden:
                hidden.append(h)

        logits = self.output_layer(h)
        if return_hidden:
            return logits, hidden
        return logits

In [ ]:
@torch.no_grad()
def init_weights(model: ResMLP, alpha: float, sigma2_block: float | None = None) -> None:
    """Initialise all weights in place with zero-mean Gaussians.

    input layer: N(0, 1/784); every block: N(0, sigma2_block), default 2(1 - alpha^2)/D;
    output layer: N(0, 1/D). Pass sigma2_block = 2/D for plain He initialisation (Section 2).

    Where the three variances come from
    -----------------------------------
    A linear layer with `n_in` inputs keeps the second moment of its output equal to that of its
    input when the weights have variance 1/n_in (the usual He rule for a bias-free linear map).
    The input layer therefore uses 1/784 on standardised pixels, so E[h_0^2] = 1.

    Inside a block the skip term adds the variance alpha^2 and the branch adds (1/2) * D * sigma^2
    (Section 3), so holding the block at ratio 1 asks for sigma^2 = 2 (1 - alpha^2) / D. With
    alpha = 0 this is the plain He value 2/D, and with alpha = 1/sqrt(2) it is 1/D, the UDL
    Fig. 11.6b rescaling.

    The output layer uses 1/D, so the logits start at the same scale as the last hidden layer for
    every alpha. That keeps the initial output scale comparable between conditions; it is not
    used by the variance argument itself.
    """
    depth_width = model.input_layer.out_features          # D
    if sigma2_block is None:
        sigma2_block = 2.0 * (1.0 - alpha ** 2) / depth_width
    if sigma2_block < 0.0:
        raise ValueError(f"alpha = {alpha} gives a negative block variance; need |alpha| <= 1")

    model.input_layer.weight.normal_(mean=0.0, std=model.input_layer.in_features ** -0.5)
    for block in model.blocks:
        block.weight.normal_(mean=0.0, std=float(sigma2_block) ** 0.5)
    model.output_layer.weight.normal_(mean=0.0, std=depth_width ** -0.5)

## 2. Reproducing UDL §11.3: variance doubling and the 1/√2 fix

Forward pass only, no training. For ≥ 10 networks (different seeds), push `DATA["x_calib"]`
through the network and record E[h_l²] / E[h_0²] for every layer l:

- (a) α = 1 with He initialisation (σ² = 2/D): expected to grow by ≈ ×2 per block;
- (b) α = 1/√2 with σ² = 1/D: expected to stay ≈ 1.

Plot mean ± sd over networks against l (log scale for (a)). Report the per-block factor as an
average over networks: a single network deviates because of finite width (Section 4).

In [ ]:
N_NETWORKS = 12       # the protocol asks for >= 10 networks
BASE_SEED = 100       # seeds 100...111: they only make the check reproducible, they are not run seeds


def relative_second_moments(model: "ResMLP", x: torch.Tensor) -> np.ndarray:
    """E[h_l^2] / E[h_0^2] for l = 0...L, on one fixed batch and with no gradients."""
    model.eval()
    with torch.no_grad():
        _, hidden = model(x, return_hidden=True)
        m2 = torch.stack([h.pow(2).mean() for h in hidden])
    return (m2 / m2[0]).cpu().numpy()


def variance_propagation(depth: int, alpha: float, sigma2_block: float, x: torch.Tensor,
                         n_networks: int = N_NETWORKS, base_seed: int = BASE_SEED):
    """One curve per network, plus the per-block factor of each network.

    The per-block factor is the geometric mean, (final ratio) ** (1 / depth). Averaged over
    networks it should be the multiplier the derivation predicts; single networks deviate because
    the width is finite (Section 4).
    """
    curves, factors = [], []
    for i in range(n_networks):
        set_seed(base_seed + i)
        model = ResMLP(depth, C.WIDTH, alpha).to(x.device)
        init_weights(model, alpha, sigma2_block=sigma2_block)
        curve = relative_second_moments(model, x)
        curves.append(curve)
        factors.append(curve[-1] ** (1.0 / depth))
    return np.stack(curves), np.array(factors)


def describe(values: np.ndarray) -> str:
    """Multiplicative spread: median and the min...max range, plus mean +- sd."""
    values = np.asarray(values, dtype=float)
    return (f"median {np.median(values):.3g}  range {values.min():.3g}...{values.max():.3g}  "
            f"({values.max() / values.min():.1f}x)  mean {values.mean():.3g} +/- {values.std(ddof=1):.3g}")


# (a) alpha = 1 with plain He (sigma^2 = 2/D): the textbook doubling
curves_he, factor_he = variance_propagation(C.DEPTH_MAIN, 1.0, 2.0 / C.WIDTH, DATA["x_calib"])
# (b) alpha = 1/sqrt(2) with sigma^2 = 1/D: the same sigma^2(alpha), i.e. He * 1/sqrt(2)
curves_fix, factor_fix = variance_propagation(C.DEPTH_MAIN, C.ALPHA_UDL, 1.0 / C.WIDTH, DATA["x_calib"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
layers = np.arange(curves_he.shape[1])
plot_mean_sd(axes[0], layers, curves_he, label="alpha = 1, He (sigma^2 = 2/D)", marker="o", ms=3)
plot_mean_sd(axes[1], layers, curves_fix, label="alpha = 1/sqrt(2), sigma^2 = 1/D", marker="o", ms=3)
axes[0].set_yscale("log")
axes[0].set_title("(a) alpha = 1 with He: about x2 per block")
axes[1].set_title("(b) alpha = 1/sqrt(2) with the 1/sqrt(2) fix: stays at 1")
axes[1].set_ylim(0.0, 2.5)
for ax in axes:
    ax.set_xlabel("layer l")
    ax.set_ylabel("E[h_l^2] / E[h_0^2]")
    ax.legend()

print(f"(a) alpha = 1, He:        per-block factor {factor_he.mean():.3f} +/- {factor_he.std(ddof=1):.3f}"
      f"   (theory 2.000)")
print(f"    final layer, {N_NETWORKS} networks: {describe(curves_he[:, -1])}")
print(f"(b) alpha = 1/sqrt(2):    per-block factor {factor_fix.mean():.3f} +/- {factor_fix.std(ddof=1):.3f}"
      f"   (theory 1.000)")
print(f"    final layer, {N_NETWORKS} networks: {describe(curves_fix[:, -1])}")
plt.tight_layout();

## 3. Deriving σ²(α)

The block is $h_{l+1} = \alpha h_l + W_l\,\mathrm{ReLU}(h_l)$ with $W_l \in \mathbb{R}^{D \times D}$,
entries i.i.d. $\mathcal N(0,\sigma^2)$, no bias. Write $q_l = \mathbb E[h_l^2]$, averaged over the
$D$ units and over the batch. Assume the units are exchangeable and $h_l$ is centred, so ReLU
keeps half of the second moment: $\mathbb E[\mathrm{ReLU}(h)^2] = \tfrac12\, q_l$.

### Forward

For one output unit $i$ the branch is $\sum_j W_{ij}\,\mathrm{ReLU}(h_j)$, so its second moment is

$$\mathbb E\Big[\big(\textstyle\sum_j W_{ij}\mathrm{ReLU}(h_j)\big)^2\Big]
 = \sum_j \sigma^2\,\mathbb E[\mathrm{ReLU}(h_j)^2]
 = \sigma^2 D \cdot \tfrac12 q_l .$$

The cross term vanishes, $\mathbb E[h\cdot W\,\mathrm{ReLU}(h)] = 0$, because $W_l$ is zero-mean and
independent of $h_l$ ($h_l$ is built from $W_{<l}$ only). So

$$q_{l+1} = \alpha^2 q_l + \tfrac12 D\sigma^2 q_l
 \quad\Longrightarrow\quad \frac{q_{l+1}}{q_l} = \alpha^2 + \tfrac12 D\sigma^2 .$$

Setting the ratio to 1 gives the initialisation the design uses:

$$\boxed{\ \sigma^2(\alpha) = \frac{2(1-\alpha^2)}{D}\ }$$

### Backward

The gradient with respect to $h_l$ is
$\delta_l = \alpha\,\delta_{l+1} + \mathrm{diag}(\mathbb 1[h_l>0])\,W_l^\top \delta_{l+1}$,
with $\delta_L = \partial \mathcal L/\partial h_L$. Write $s_l = \mathbb E[\delta_l^2]$ per unit.

The branch part of one unit $j$ is $\mathbb 1[h_j>0]\sum_i W_{ij}\delta_{l+1,i}$, hence

$$\mathbb E\big[\text{branch}_j^2\big]
 = \mathbb 1[h_j>0]\,\sigma^2 \sum_i \mathbb E[\delta_{l+1,i}^2]
 = \mathbb 1[h_j>0]\,\sigma^2 D\, s_{l+1},$$

and averaging the indicator over units gives a factor $\tfrac12$. The cross term is zero for the
same reason as in the forward pass: $\delta_{l+1}$ depends on $W_{l+1},\dots$ but not on $W_l$, and
$\mathbb E[W_{ij}] = 0$. Therefore

$$s_l = \big(\alpha^2 + \tfrac12 D\sigma^2\big)\, s_{l+1},$$

exactly the same factor as the forward pass. This is why one initialisation fixes both directions,
and why no separate backward rescaling is needed. It is also why $W$ being square ($D\to D$)
matters here: the sum over $i$ runs over $D$ terms in both directions.

### Special cases

| $\alpha$ | $\sigma^2(\alpha)$ | per-block ratio $\alpha^2 + \tfrac12 D\sigma^2$ | what it is |
|---|---|---|---|
| $0$ | $2/D$ | $1$ | plain deep net with He initialisation (BL5) |
| $1/\sqrt2$ | $1/D$ | $1$ | He $\times\,1/\sqrt2$, the UDL Fig. 11.6b rescaling |
| $1$ with He | $2/D$ | $2$ | the variance doubling of UDL §11.3 |

Two remarks we need later.

- The derivation is in expectation over the weights. A single finite-width network misses the
  ratio by a random amount, and those errors multiply over 32 blocks (Section 4). That is what
  the calibration step is for.
- The result only holds because the block is pre-activation — ReLU first, then the linear map —
  as in UDL Fig. 11.5b. It is also why the model has no bias and no normalisation: a bias would
  break the zero-mean assumption, and normalisation would set the variance by force instead of by
  the initialisation we are testing.

## 4. Finite-width spread and calibration

The derivation holds in expectation over the weights. Show how much single networks deviate
(final-layer variance across ≥ 10 networks, for each α in `C.ALPHAS[32]`), then calibrate each
network so its forward variance is exactly preserved on the calibration batch.

Checks to report for every condition (both depths):
1. forward: per-layer E[h_l²] ratio after calibration = 1.000 (and close to 1 on held-out data);
2. backward: per-layer second moment of ∂loss/∂h_l compared across conditions
   (hint: `h.retain_grad()` on the hidden states returned by `forward(..., return_hidden=True)`).

In [ ]:
@torch.no_grad()
def calibrate(model: ResMLP, x_calib: torch.Tensor, alpha: float) -> list[float]:
    """Rescale each block's W so that E[h_{l+1}^2] = E[h_l^2] exactly on x_calib.

    For block l with f = W_l(ReLU(h_l)):  a = E[f^2], b = E[h_l * f], c = E[h_l^2]
        s = (-alpha * b + sqrt(alpha^2 * b^2 + (1 - alpha^2) * a * c)) / a,   W_l <- s * W_l
    Process the blocks in order, so block l+1 sees the already-calibrated h_{l+1}.
    Returns the list of scale factors s (they should all be close to 1).

    The formula is the positive root of alpha^2 c + 2 alpha s b + s^2 a = c, i.e. of
    E[(alpha h + s f)^2] = E[h^2]. It is the LSUV idea with the skip term kept: we do not want a
    block-wise unit variance, we want the *ratio* to stay 1 at this alpha.
    Only the forward pass is matched here, so the backward pass is checked separately (Section 4).
    """
    model.eval()
    h = model.input_layer(x_calib.flatten(start_dim=1))
    scales: list[float] = []

    for block in model.blocks:
        f = block(torch.relu(h))
        # reduce on the device, then finish the arithmetic in Python floats (double):
        # MPS has no float64, and the scalars are small so the cost is irrelevant.
        a = float(f.pow(2).mean())
        b = float((h * f).mean())
        c = float(h.pow(2).mean())
        if not (a > 0.0) or a != a:
            raise ValueError("the residual branch is zero (or not finite); nothing to rescale")

        root = ((alpha * b) ** 2 + (1.0 - alpha ** 2) * a * c) ** 0.5
        s = (-alpha * b + root) / a

        block.weight.mul_(s)
        h = alpha * h + s * f          # the next block must see the rescaled output
        scales.append(s)

    return scales

In [ ]:
def final_layer_variance(model: "ResMLP", x: torch.Tensor) -> float:
    """E[h_L^2] on one batch (no gradients): the scale that reaches the output layer."""
    model.eval()
    with torch.no_grad():
        _, hidden = model(x, return_hidden=True)
        return float(hidden[-1].pow(2).mean())


def build(alpha: float, depth: int = C.DEPTH_MAIN, seed: int = BASE_SEED,
          calibrate_it: bool = True) -> "ResMLP":
    """A fresh network for one condition: seed -> init -> (optionally) calibrate."""
    set_seed(seed)
    model = ResMLP(depth, C.WIDTH, alpha).to(DATA["x_calib"].device)
    init_weights(model, alpha)
    if calibrate_it:
        calibrate(model, DATA["x_calib"], alpha)
    return model


def gradient_second_moments(model: "ResMLP", x: torch.Tensor, y: torch.Tensor):
    """E[(d loss / d h_l)^2] for l = 0...L, and the loss value, on one batch."""
    model.eval()
    model.zero_grad(set_to_none=True)
    logits, hidden = model(x, return_hidden=True)
    for h in hidden:
        h.retain_grad()
    loss = nn.functional.cross_entropy(logits, y)
    loss.backward()
    grads = torch.stack([h.grad.detach().pow(2).mean() for h in hidden]).cpu().numpy()
    loss_value = float(loss.detach())
    model.zero_grad(set_to_none=True)
    return grads, loss_value


x_bwd, y_bwd = DATA["x_train"][:512], DATA["y_train"][:512]   # a fixed batch for the backward check
rows = []
for alpha in C.ALPHAS[C.DEPTH_MAIN]:
    spread, dev_calib, held_final, bwd_factor, bwd_level = [], [], [], [], []
    for i in range(N_NETWORKS):
        seed = BASE_SEED + i

        raw = build(alpha, seed=seed, calibrate_it=False)          # before calibration
        spread.append(final_layer_variance(raw, DATA["x_calib"]))

        model = build(alpha, seed=seed)                            # after calibration
        cal = relative_second_moments(model, DATA["x_calib"])
        dev_calib.append(float(np.abs(cal - 1.0).max()))
        held_final.append(float(relative_second_moments(model, DATA["x_val"][:512])[-1]))

        grads, _ = gradient_second_moments(model, x_bwd, y_bwd)
        bwd_factor.append((grads[0] / grads[-1]) ** (1.0 / C.DEPTH_MAIN))
        bwd_level.append(grads[-1])

    spread = np.array(spread)
    rows.append({
        "alpha": alpha,
        "spread": spread.max() / spread.min(),      # how far single networks drift apart
        "dev_calib": float(np.max(dev_calib)),      # worst per-layer deviation from 1
        "held_final": float(np.mean(held_final)),   # forward ratio on data it never saw
        "bwd_factor": float(np.mean(bwd_factor)),
        "bwd_level": float(np.mean(bwd_level)),
    })

print(f"{'alpha':>6} | {'E[h_L^2] max/min':>16} | {'worst |ratio-1|':>15} | "
      f"{'held-out E[h_L^2]/E[h_0^2]':>26} | {'bwd per-block':>13} | {'E[dL/dh_L]^2':>12}")
print("-" * 100)
for r in rows:
    print(f"{r['alpha']:6.4f} | {r['spread']:16.2f} | {r['dev_calib']:15.2e} | "
          f"{r['held_final']:26.4f} | {r['bwd_factor']:13.4f} | {r['bwd_level']:12.2e}")

levels = np.array([r["bwd_level"] for r in rows])
print(f"\nacross the six conditions, E[(dL/dh_L)^2] spans {levels.min():.2e} ... {levels.max():.2e} "
      f"(largest/smallest = {levels.max() / levels.min():.2f})")


### What the checks showed (local run, to be repeated on the lab machine)

These numbers come from one local run on Apple MPS (torch 2.15, 12 networks per condition, seeds
100–111, the 512 calibration images). They are a sanity check for the code, not report results.

| Check | Result |
|---|---|
| §2(a): α = 1 with He, σ² = 2/D | per-block factor 1.993 ± 0.045 (theory 2.000); final layer median 3.2×10⁹, which is the expected 2³² ≈ 4×10⁹ up to finite-width noise |
| §2(b): α = 1/√2, σ² = 1/D | per-block factor 0.997 ± 0.023 (theory 1.000); final layer median 0.74, range 0.41 … 5.1 |
| §4: spread of E[h_L²] over 12 networks, before calibration | 1.6× at α = 0.97 up to 26× at α = 0.5 — single networks drift apart, and less so as α grows |
| §4: forward ratio after calibration | worst \|ratio − 1\| ≈ 2×10⁻⁶, i.e. 1.000 at printed precision (that residual is float32 rounding) |
| §4: forward ratio on held-out data | 0.985 … 1.003 — the calibration transfers past the 512 images it used |
| §4: backward | per-block factor 1.001 … 1.014 at every α, and E[(∂L/∂h_L)²] agrees across conditions within 5% |

How to read this.

- §2 reproduces both textbook claims: ×2 per block with α = 1 and He, and a flat curve once the
  branch is rescaled to σ² = 1/D at α = 1/√2.
- §2(b) also shows why an initialisation alone is not enough. The *average* per-block factor is
  1.000, but the 32 factors multiply, so one network can end up at 0.41 and the next at 5.1 with
  the same α. That is finite-width drift, and it is what the calibration step removes.
- §4 shows the calibration does what it is supposed to: the forward ratio is 1.000 on the
  calibration batch, still ≈ 1 on held-out data, and the backward pass — which calibration never
  touches — is already matched by the derived σ²(α) at every α.
- That combination is the control the rest of the project rests on: after this step the signal
  scale is the same for every α, so any training difference we later see cannot be blamed on it.


## 5. Training the baseline (32 blocks, α = 1/√2)

Protocol: sweep `C.LR_GRID` with `C.TUNE_SEED`, select the learning rate by final training loss
(rule in `hypotheses.md`), then rerun with `C.FINAL_SEEDS` and the test set.

QUICK mode target: load cached runs and train one small live example.

In [ ]:
def train_one_run(depth: int, alpha: float, lr: float, seed: int, epochs: int = C.EPOCHS,
                  width: int = C.WIDTH, data: dict = None, save: bool = True,
                  with_test: bool = False, n_train: int | None = None, tag: str = "") -> dict:
    """Train one network; return (and by default save) its result record.

    set_seed(seed) -> ResMLP(depth, width, alpha) -> init_weights -> calibrate on data["x_calib"]
    -> torch.optim.SGD(lr=lr, momentum=C.MOMENTUM), batches of C.BATCH_SIZE shuffled with
       make_generator(seed)
    -> after every epoch: evaluate() on the full training and validation sets
    -> if the loss becomes NaN/inf, stop and set diverged = True (keep the record).

    Record keys: depth, width, alpha, lr, seed, epochs, train_loss, val_loss, train_acc, val_acc
    (lists per epoch; train_loss[C.EARLY_EPOCH - 1] is the early-training metric),
    final_train_loss, diverged, seconds, and test_loss / test_acc when
    with_test (final runs only). n_train trains on a subset (QUICK live example; then save=False).
    """
    data = DATA if data is None else data
    # TODO
    raise NotImplementedError

In [ ]:
if QUICK:
    baseline = load_results(depth=C.DEPTH_MAIN, alpha=C.ALPHA_BASELINE)
    print(f"loaded {len(baseline)} cached baseline runs")
    # TODO: one small live example, e.g. train_one_run(..., epochs=1, n_train=10_000, save=False)
else:
    pass  # TODO: lr sweep (skip runs where result_exists), select lr, final seeds with with_test=True

## 6. Baseline results and variability

_TODO:_ training-loss curves (mean ± sd over the final seeds), final loss mean ± sd, and a short
comment on the spread. State which parts of the experiment the random state affects (weight
initialisation and data order) and which are fixed (data split and calibration batch).

In [ ]:
# TODO: plots and summary table (summarise(...) from utils)

## 7. Decisions carried into NB2

The rules below are fixed here; the numbers that need runs are filled in when those runs land.
The baseline runs give the epoch check and τ; the range check is a separate run. Then
`hypotheses.md` is frozen and its commit hash goes into NB2 §0.

1. **Has the baseline loss flattened by epoch `C.EPOCHS` (= 6)?**
   Rule: compare the seed-averaged training loss at epoch 4 and at epoch 6. If the drop from 4 to 6
   is under 5% of the epoch-1 → 6 drop, the curve counts as flat and 6 epochs stay. If not, we go to
   12 epochs before the main sweep and write that down.
   *Number:* pending, from the NB1 §5 baseline runs.

2. **τ, the usable-learning-rate threshold.**
   Rule: τ = 1.25 × L_base, where L_base is the baseline final training loss at the selected
   learning rate (seed 0, 32 blocks, α = 1/√2). A learning rate is usable if the run does not
   diverge and its final training loss is ≤ τ.
   *Number:* pending the same baseline runs; it is written into `hypotheses.md` at the freeze gate.

3. **Range check: are α = 0.5 and α = 0.9 distinguishable at 32 blocks?**
   Protocol: 32 blocks, α ∈ {0.5, 0.9}, two or three learning rates. We call them distinguishable
   if the 95% t interval of the difference of the seed means excludes 0 at epoch `C.EARLY_EPOCH`
   or at epoch `C.EPOCHS`. If they are not distinguishable, we discuss whether 32 blocks is deep
   enough before freezing the design. This only checks the measuring range; it is not evidence for
   any hypothesis.
   *Number:* pending, from the range-check runs.

What this section already settles: the three answers have a written rule, so nobody can choose the
rule after seeing the main results. Sections 2 and 4 are the support for the control we rely on
everywhere else — after calibration the forward ratio is 1.000 and the backward scale is the same
for every α, so any difference we later see in training cannot be blamed on the signal scale.

In [ ]:
print(f"Total runtime: {time.time() - T0:.0f} s  (QUICK = {QUICK}; must stay under 300 s when QUICK)")